# Maia Corpus v8 — Production Canary v1 (60 complete chunks)

**CPU only.** Process 10 complete `.jsonl.gz` chunks from each of six sources (60 total). This is a **real-data canary**, not full production. Uses the **same pipeline implementation** as Production Preparation v1, with a separate immutable output directory. The source corpus and scientific freeze are read-only.

**Safety:** no overwrite of pilot or production; source metadata validated; SQLite transaction per chunk; committed shard checksums; recovery quarantine for uncommitted files; periodic JSON progress; explicit stop after 60 chunks. No model training. Run in Colab with CPU and Drive mounted.

**Important:** This canary measures *this Colab + Drive environment*, not sustained A100 throughput. It does not certify licensing, near duplicates, full freeze SHA, or long-term production reliability.

In [ ]:
# 1. Mount Drive (Colab only)
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; use locally mounted paths.')

In [ ]:
# 2. Dependencies and environment
import sys, subprocess
try:
    import transformers, tokenizers, numpy
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<5', 'tokenizers>=0.20', 'numpy>=1.26'])
import os, json, gzip, hashlib, sqlite3, time, shutil, platform, random, collections, struct
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from transformers import AutoTokenizer
print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('Transformers:', __import__('transformers').__version__)

In [ ]:
# 3. Configuration — review before running
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
FREEZE_MANIFEST = ROOT / 'freeze_v8_001/manifest.json'
AUDIT_CONFIG = ROOT / 'quality_audit_full_with_tokenizer_v1/audit_config.json'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
# Optional: add the exact directory that contains original .jsonl.gz files.
# The notebook searches these roots recursively; avoid scanning unrelated large trees.
SEARCH_ROOTS = [ROOT]

# Always use a NEW output directory for a changed configuration.
OUT = ROOT / 'pretraining_canary_v1_60chunks'
PRODUCTION_MODE = True   # Full-document processing for selected canary chunks only.
MAX_CHUNKS_PER_SOURCE_PILOT = 10
MAX_DOCS_PER_CHUNK_PILOT = 100
BLOCK_SIZE = 2048
SHARD_MAX_BLOCKS = 512   # ~2 MiB of token IDs per shard at uint32
SPLIT_PERCENT = {'train': 94, 'validation': 3, 'test': 3}
SPLIT_SEED = 'maia-v8-split-v1'
EOS_PER_DOCUMENT = True
SOURCE_NAMES = ['fineweb2_es','fineweb2_pt','fineweb_edu_en','wikipedia_en','wikipedia_es','wikipedia_pt']
LANGUAGE = {'fineweb2_es':'es','fineweb2_pt':'pt','fineweb_edu_en':'en','wikipedia_en':'en','wikipedia_es':'es','wikipedia_pt':'pt'}
LANGUAGE_TARGET = {'pt':0.40,'en':0.35,'es':0.25}
TARGET_TRAIN_EXPOSURES = 10_100_000_000
# If an existing output directory was made with another config, the notebook aborts.
assert sum(SPLIT_PERCENT.values()) == 100
assert MAX_CHUNKS_PER_SOURCE_PILOT == 10
print('MODE:', 'PRODUCTION' if PRODUCTION_MODE else 'PILOT')
print('OUTPUT:', OUT)
# Guardrail: PRODUCTION_MODE=True disables the 100-doc pilot cap,
# but selection below ALWAYS limits input to exactly 10 chunks/source.


In [ ]:
# 4. Validate source freeze and tokenizer fingerprint
assert FREEZE_MANIFEST.is_file(), f'Missing freeze: {FREEZE_MANIFEST}'
freeze_bytes = FREEZE_MANIFEST.read_bytes()
freeze = json.loads(freeze_bytes)
freeze_sha = hashlib.sha256(freeze_bytes).hexdigest()
assert TOKENIZER_DIR.is_dir(), f'Missing tokenizer: {TOKENIZER_DIR}'
TOK_FILES = ['merges.txt','tokenizer_config.json','vocab.json','tokenizer.json']
for filename in TOK_FILES:
    assert (TOKENIZER_DIR/filename).is_file(), f'Missing tokenizer file: {filename}'
file_hashes = {f:hashlib.sha256((TOKENIZER_DIR/f).read_bytes()).hexdigest() for f in TOK_FILES}
# These hashes are taken from the successful stratified pilot v2.
EXPECTED_HASHES = {
 'merges.txt':'b4b5c08bd27684fb3c6426ffe33adaaf7375340c733dad2f1d609d90767cef8a',
 'tokenizer_config.json':'a1cf74974f6d8a98aef2b7a2bb34084a6ad9d79c4e36e7280d369628149c7062',
 'vocab.json':'63b18c80a572b14e90de3529ac068b76ca49e449b7c4219eccbc05d6cee4cf6c',
 'tokenizer.json':'fd6cc1c4c1a487c5c4b45ee812762c471e5776ff1845fa48ad04c245ae4644a6'
}
assert file_hashes == EXPECTED_HASHES, 'Tokenizer files differ from validated pilot v2.'
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True)
assert len(tok) == 50257, f'Unexpected vocab size: {len(tok)}'
assert tok.eos_token_id == 2, f'Unexpected EOS: {tok.eos_token_id}'
assert tok.pad_token_id == 1, f'Unexpected PAD: {tok.pad_token_id}'
print('Freeze manifest SHA-256:', freeze_sha)
print('Tokenizer file hashes: MATCH pilot v2')
print('Historical tokenizer identity: NOT PROVEN')

In [ ]:
# 5. Discover input chunks and create a deterministic inventory
# Discovery may take a while on Google Drive. Paths are sorted deterministically.
from collections import defaultdict
inventory = defaultdict(list)
for search_root in SEARCH_ROOTS:
    assert Path(search_root).exists(), f'Search root missing: {search_root}'
    for path in Path(search_root).rglob('*.jsonl.gz'):
        for source in SOURCE_NAMES:
            if path.name.startswith(source + '_chunk_'):
                inventory[source].append(path)
                break
for source in SOURCE_NAMES:
    inventory[source] = sorted(set(inventory[source]), key=lambda p:str(p))
    print(f'{source:18} {len(inventory[source]):6} chunks')
    assert inventory[source], f'No source chunks found: {source}. Set SEARCH_ROOTS explicitly.'
selected = {source:inventory[source][:MAX_CHUNKS_PER_SOURCE_PILOT] for source in SOURCE_NAMES}
assert all(len(selected[s]) == 10 for s in SOURCE_NAMES), 'Canary requires exactly 10 chunks/source.'
selected_paths = [p for source in SOURCE_NAMES for p in selected[source]]
assert len(set(map(str,selected_paths))) == len(selected_paths), 'Duplicate file paths in inventory.'
print('Selected chunks:', len(selected_paths))
assert len(selected_paths) == 60, 'CANARY SAFETY: selected input must be exactly 60 chunks.'


In [ ]:
# 6. Immutable run configuration and SQLite state
# File sizes/mtime guard accidental source changes. For full source cryptographic
# verification, use the audited inventory/checksums and an independent full hash pass.
config = {
 'schema_version':'1.0', 'production_mode':PRODUCTION_MODE, 'block_size':BLOCK_SIZE,
 'shard_max_blocks':SHARD_MAX_BLOCKS, 'split_percent':SPLIT_PERCENT,
 'split_seed':SPLIT_SEED, 'eos_per_document':EOS_PER_DOCUMENT,
 'source_names':SOURCE_NAMES, 'language':LANGUAGE,
 'tokenizer_file_sha256':file_hashes, 'freeze_manifest_sha256':freeze_sha,
 'max_docs_per_chunk_pilot':None if PRODUCTION_MODE else MAX_DOCS_PER_CHUNK_PILOT,
 'selected_inventory':[
  {'source':s,'path':str(p),'size':p.stat().st_size,'mtime_ns':p.stat().st_mtime_ns}
  for s in SOURCE_NAMES for p in selected[s]
 ]
}
config_digest = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
OUT.mkdir(parents=True,exist_ok=True)
config_path = OUT/'run_config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == config, 'CONFIG CHANGED: choose a NEW OUT directory.'
else:
    with config_path.open('x',encoding='utf-8') as f: json.dump(config,f,indent=2,ensure_ascii=False)
DB = OUT/'state.sqlite3'
con = sqlite3.connect(DB)
con.execute('PRAGMA journal_mode=DELETE')  # safer on mounted Drive than WAL
con.execute('PRAGMA synchronous=FULL')
con.executescript('''
CREATE TABLE IF NOT EXISTS seen(hash TEXT PRIMARY KEY, split TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS completed_chunk(path TEXT PRIMARY KEY, source TEXT NOT NULL, docs INTEGER NOT NULL, duplicates INTEGER NOT NULL, tokens INTEGER NOT NULL, completed_utc TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS shard(source TEXT NOT NULL, split TEXT NOT NULL, shard_index INTEGER NOT NULL, path TEXT NOT NULL UNIQUE, blocks INTEGER NOT NULL, sha256 TEXT NOT NULL, PRIMARY KEY(source,split,shard_index));
CREATE TABLE IF NOT EXISTS pending(source TEXT NOT NULL, split TEXT NOT NULL, ids BLOB NOT NULL, PRIMARY KEY(source,split));
''')
con.commit()
print('Run config SHA-256:',config_digest)
print('State:',DB)

## Resume strategy

Each input chunk is processed as **one SQLite transaction**. All exact-hash dedup records, output shard metadata, pending partial blocks, and completed-chunk state commit together. Token shards are written to temporary files, atomically renamed into place, then the transaction commits. If the process crashes before commit, orphan shard files can remain; on resume the notebook verifies their hash against committed metadata and **refuses to proceed if an unexpected output filename exists**. A manual recovery is required for such an orphan (do not delete it without inspection). The process is single-writer only.

**Memory:** only the current chunk, token buffers and tokenizer are kept in memory. Hashes live in SQLite; on 10M documents this database may be large and slow on Drive. For production, a local SSD staging directory with verified copy-back is strongly recommended.

**Split policy:** SHA-256 of normalized document text, salted with a fixed seed, assigns documents to train/validation/test. Exact identical texts are assigned to the same split, and subsequent copies are skipped globally. This does **not** prevent near-duplicate or cross-document leakage.

In [ ]:
# 7. Helpers: deterministic split, atomic shards and integrity checks
import unicodedata

def canonical_text(value):
    return unicodedata.normalize('NFC',value).strip()

def document_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def split_for_hash(h):
    bucket = int.from_bytes(hashlib.sha256((SPLIT_SEED + ':' + h).encode()).digest()[:8],'big') % 10000
    if bucket < SPLIT_PERCENT['train']*100: return 'train'
    if bucket < (SPLIT_PERCENT['train']+SPLIT_PERCENT['validation'])*100: return 'validation'
    return 'test'

def shard_file(source,split,index):
    return OUT/'shards'/source/split/f'{source}_{split}_{index:06d}.bin'

def verify_committed_shards():
    for source,split,idx,p,blocks,sha in con.execute('SELECT source,split,shard_index,path,blocks,sha256 FROM shard'):
        path=Path(p)
        assert path.is_file(), f'Missing committed shard: {path}'
        assert path.stat().st_size == blocks*BLOCK_SIZE*4, f'Unexpected shard size: {path}'
        assert hashlib.sha256(path.read_bytes()).hexdigest() == sha, f'Shard hash mismatch: {path}'
    print('Committed shard integrity: PASS')

verify_committed_shards()

In [ ]:
# 8. Process a chunk atomically, retaining tails between chunks
# Shards use little-endian uint32 token IDs, exactly BLOCK_SIZE IDs per block.
# Each (source,split) has its own token stream; streams never mix languages/splits.
# IMPORTANT: This cell only defines functions; the next cell performs processing.

def process_chunk(source,path):
    path = Path(path)
    row=con.execute('SELECT 1 FROM completed_chunk WHERE path=?',(str(path),)).fetchone()
    if row: return 'SKIP_COMPLETED'
    # Check source metadata against the immutable run config.
    record = next(x for x in config['selected_inventory'] if x['path']==str(path))
    st=path.stat()
    assert (st.st_size,st.st_mtime_ns)==(record['size'],record['mtime_ns']), f'Source changed: {path}'
    # Prepare pending stream state in RAM. Existing SQLite state is not changed until commit.
    buffers={}
    for split in SPLIT_PERCENT:
        row=con.execute('SELECT ids FROM pending WHERE source=? AND split=?',(source,split)).fetchone()
        buffers[split]=np.frombuffer(row[0],dtype='<u4').astype('<u4').tolist() if row else []
    new_hashes=[]; local_hashes=set(); docs=dups=tokens=0
    new_shards=[]
    try:
        with gzip.open(path,'rt',encoding='utf-8') as stream:
            for line in stream:
                if not line.strip(): continue
                obj=json.loads(line)
                raw=obj.get('text')
                if not isinstance(raw,str): raise ValueError(f'Missing text in {path}')
                txt=canonical_text(raw)
                if not txt: continue
                h=document_hash(txt)
                if h in local_hashes or con.execute('SELECT 1 FROM seen WHERE hash=?',(h,)).fetchone():
                    dups+=1; continue
                local_hashes.add(h)
                split=split_for_hash(h)
                ids=tok.encode(txt,add_special_tokens=False)
                if EOS_PER_DOCUMENT: ids.append(tok.eos_token_id)
                assert all(0 <= t < len(tok) for t in ids)
                buffers[split].extend(ids)
                new_hashes.append((h,split)); docs+=1; tokens+=len(ids)
                if not PRODUCTION_MODE and docs+dups>=MAX_DOCS_PER_CHUNK_PILOT: break
        # Prepare full blocks for each split, and preserve incomplete tails.
        for split in SPLIT_PERCENT:
            buf=buffers[split]
            nfull=len(buf)//BLOCK_SIZE
            if nfull:
                previous=con.execute('SELECT COALESCE(MAX(shard_index),-1) FROM shard WHERE source=? AND split=?',(source,split)).fetchone()[0]
                for start_block in range(0,nfull,SHARD_MAX_BLOCKS):
                    take=min(SHARD_MAX_BLOCKS,nfull-start_block)
                    index=previous+1+len([x for x in new_shards if x[0]==source and x[1]==split])
                    target=shard_file(source,split,index)
                    assert not target.exists(), f'Uncommitted/orphan shard exists: {target}; investigate before resuming.'
                    target.parent.mkdir(parents=True,exist_ok=True)
                    arr=np.asarray(buf[start_block*BLOCK_SIZE:(start_block+take)*BLOCK_SIZE],dtype='<u4')
                    payload=arr.tobytes()
                    temp=target.with_suffix('.bin.tmp')
                    assert not temp.exists(), f'Orphan temporary shard: {temp}'
                    with temp.open('xb') as f:
                        f.write(payload); f.flush(); os.fsync(f.fileno())
                    os.replace(temp,target)
                    new_shards.append((source,split,index,str(target),take,hashlib.sha256(payload).hexdigest()))
            buffers[split]=buf[nfull*BLOCK_SIZE:]
            assert len(buffers[split])<BLOCK_SIZE
        with con:
            con.executemany('INSERT INTO seen(hash,split) VALUES(?,?)',new_hashes)
            con.executemany('INSERT INTO shard(source,split,shard_index,path,blocks,sha256) VALUES(?,?,?,?,?,?)',new_shards)
            for split in SPLIT_PERCENT:
                tail=np.asarray(buffers[split],dtype='<u4').tobytes()
                con.execute('INSERT INTO pending(source,split,ids) VALUES(?,?,?) ON CONFLICT(source,split) DO UPDATE SET ids=excluded.ids',(source,split,tail))
            con.execute('INSERT INTO completed_chunk(path,source,docs,duplicates,tokens,completed_utc) VALUES(?,?,?,?,?,?)',
                        (str(path),source,docs,dups,tokens,datetime.now(timezone.utc).isoformat()))
        return f'OK docs={docs} dup={dups} tokens={tokens} shards={len(new_shards)}'
    except Exception:
        # Never silently remove generated files: inspect orphans on recovery.
        raise

## Crash recovery and orphan quarantine

A process interruption can leave a fully written shard whose SQLite transaction did not commit. The following step **never deletes** such files. It verifies every committed shard first, then moves only unreferenced `.bin` or `.bin.tmp` files **inside this canary's output directory** into a timestamped quarantine directory. After quarantine, `process_chunk` may safely retry an incomplete chunk. Do not run two copies of this notebook against the same output directory simultaneously.

In [ ]:
# Verify committed shards and quarantine ONLY uncommitted canary files.
from datetime import datetime, timezone
verify_committed_shards()
assert OUT.name == 'pretraining_canary_v1_60chunks', 'Refusing recovery outside canary output.'
committed = {str(Path(p).resolve()) for (p,) in con.execute('SELECT path FROM shard')}
shards_root = OUT/'shards'
orphans = []
if shards_root.exists():
    for p in sorted(shards_root.rglob('*')):
        if p.is_file() and (p.name.endswith('.bin') or p.name.endswith('.bin.tmp')):
            if str(p.resolve()) not in committed:
                orphans.append(p)
if orphans:
    quarantine = OUT/'quarantine'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    for p in orphans:
        dest = quarantine / p.relative_to(shards_root)
        dest.parent.mkdir(parents=True,exist_ok=True)
        assert not dest.exists()
        os.replace(p,dest)
    print('QUARANTINED uncommitted canary files:',len(orphans),'to',quarantine)
else:
    print('No uncommitted canary shard files found.')
print('Already completed chunks:',con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0])


## Execute canary (up to 60 complete chunks; resumable)

In [ ]:
# Execute all 60 selected complete chunks, with resume and per-chunk progress.
# If Colab disconnects, rerun the notebook from the beginning with the same configuration.
import time, json
progress_dir=OUT/'progress';progress_dir.mkdir(parents=True,exist_ok=True)
started=time.perf_counter()
new_chunks=0
for source in SOURCE_NAMES:
    for idx,path in enumerate(selected[source],1):
        t0=time.perf_counter()
        result=process_chunk(source,path)
        elapsed=time.perf_counter()-t0
        if result != 'SKIP_COMPLETED': new_chunks+=1
        totals=con.execute('SELECT COALESCE(SUM(tokens),0),COUNT(*),COALESCE(SUM(docs),0),COALESCE(SUM(duplicates),0) FROM completed_chunk').fetchone()
        report={'source':source,'source_index':idx,'path':str(path),'result':result,'seconds':round(elapsed,3),
                'total_tokens':totals[0],'completed_chunks':totals[1],'total_documents':totals[2],
                'total_duplicates':totals[3],'timestamp_utc':datetime.now(timezone.utc).isoformat()}
        print(f"[{totals[1]:02d}/60] {source} {idx:02d}/10 | {result} | {elapsed:.1f}s | cumulative {totals[0]:,} tokens",flush=True)
        # Atomic progress snapshot, overwritten intentionally (not source or manifest).
        progress_file=progress_dir/'latest.json';tmp=progress_dir/'latest.json.tmp'
        tmp.write_text(json.dumps(report,indent=2),encoding='utf-8');os.replace(tmp,progress_file)
assert con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0] == 60, 'Canary incomplete.'
print('Canary processing complete. New chunks this session:',new_chunks)
print('Session elapsed seconds:',round(time.perf_counter()-started,2))


In [ ]:
# 10. Validate shard integrity, disjoint splits and token accounting
verify_committed_shards()
print('SQLite quick_check:',con.execute('PRAGMA quick_check').fetchone()[0])
assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
assert con.execute('SELECT COUNT(*) FROM seen').fetchone()[0] == con.execute('SELECT COALESCE(SUM(docs),0) FROM completed_chunk').fetchone()[0]
# Every hash is assigned once, with a single split.
assert con.execute('SELECT COUNT(*) FROM (SELECT hash FROM seen GROUP BY hash HAVING COUNT(*)>1)').fetchone()[0] == 0
summary=[]
for source in SOURCE_NAMES:
    for split in SPLIT_PERCENT:
        docs=con.execute('SELECT COUNT(*) FROM seen WHERE split=? AND hash IN (SELECT hash FROM seen WHERE split=?)',(split,split)).fetchone()[0] if False else None
        blocks=con.execute('SELECT COALESCE(SUM(blocks),0) FROM shard WHERE source=? AND split=?',(source,split)).fetchone()[0]
        row=con.execute('SELECT ids FROM pending WHERE source=? AND split=?',(source,split)).fetchone()
        tail=len(row[0])//4 if row else 0
        summary.append({'source':source,'language':LANGUAGE[source],'split':split,'full_blocks':blocks,'tail_tokens':tail,'materialized_tokens':blocks*BLOCK_SIZE+tail})
assert sum(x['materialized_tokens'] for x in summary)==con.execute('SELECT COALESCE(SUM(tokens),0) FROM completed_chunk').fetchone()[0], 'Token conservation failed'
print('Token conservation: PASS')
for row in summary: print(row)

In [ ]:
# 11. Write manifest atomically, without overwriting a previous result
# For resumed runs, create a NEW timestamped manifest; never overwrite previous ones.
completed_count=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
expected_count=sum(map(len,selected.values()))
complete=completed_count==expected_count
manifest={
 'schema_version':'1.0', 'created_utc':datetime.now(timezone.utc).isoformat(),
 'run_config_sha256':config_digest,'freeze_manifest_sha256':freeze_sha,
 'production_mode':PRODUCTION_MODE,'completed_chunks':completed_count,
 'expected_chunks':expected_count,'all_selected_chunks_completed':complete,
 'source_summary':summary,'total_materialized_tokens':sum(x['materialized_tokens'] for x in summary),
 'total_completed_docs':con.execute('SELECT COALESCE(SUM(docs),0) FROM completed_chunk').fetchone()[0],
 'total_skipped_exact_duplicates':con.execute('SELECT COALESCE(SUM(duplicates),0) FROM completed_chunk').fetchone()[0],
 'shard_format':'little-endian uint32; each shard is full 2048-token blocks',
 'tails':'Pending partial blocks are stored in state.sqlite3 and are NOT exported as training blocks',
 'training_exposure_target':TARGET_TRAIN_EXPOSURES,'language_targets':LANGUAGE_TARGET,
 'notes':['This is a shard preparation manifest, not a training readiness certificate.',
          'The training sampler must operate on TRAIN shards only; validation/test are never repeated.',
          'No license, near-deduplication, or semantic leakage clearance is implied.',
          'Historical tokenizer provenance is unproven.']
}
manifest_dir=OUT/'manifests';manifest_dir.mkdir(exist_ok=True)
manifest_path=manifest_dir/('manifest_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'.json')
with manifest_path.open('x',encoding='utf-8') as f: json.dump(manifest,f,indent=2,ensure_ascii=False)
print('Manifest:',manifest_path)
print('Completed:',complete)
print('Materialized tokens:',manifest['total_materialized_tokens'])

## Training sampler policy (separate from shard preparation)

The production trainer must select **only train shards**, with a **40% PT / 35% EN / 25% ES target by actual consumed tokens**. It should maintain separate source cursors and deterministic shuffles per epoch, checkpoint the sampler RNG/cursors together with model/optimizer/scheduler states, and record exact consumed tokens. A 10.1B exposure target can involve repeats of train data; **do not** repeat validation/test. The training plan must account for incomplete tail blocks, EOS, quality exclusions, and the real train token count.

**Do not start a 10.1B-token run just because this notebook passes.** First perform: source licensing review, near-duplicate/leakage analysis, real-data GPU throughput, training resume regression, and storage capacity check. For 7.6B uint32 tokens, raw token IDs alone need ~30.4 GB (decimal), plus filesystem/database overhead; the SQLite hash index can be sizable.


In [ ]:
# 12. A separate, non-destructive sampler PLAN (no physical duplication)
# This is a plan, not a ready-to-run sampler. The actual trainer must enforce it.
train_tokens_by_lang=collections.Counter()
for item in summary:
    if item['split']=='train': train_tokens_by_lang[item['language']]+=item['materialized_tokens']
plan={
 'model':'Maia Lite 0.3.0', 'parameters':503868672, 'context':BLOCK_SIZE,
 'target_train_exposures':TARGET_TRAIN_EXPOSURES,
 'target_tokens_per_parameter':TARGET_TRAIN_EXPOSURES/503868672,
 'language_target_shares':LANGUAGE_TARGET,
 'target_exposures_by_language':{lang:round(TARGET_TRAIN_EXPOSURES*share) for lang,share in LANGUAGE_TARGET.items()},
 'available_train_tokens_by_language_in_current_run':dict(train_tokens_by_lang),
 'requires_training_sampler_implementation':True,
 'source_manifest':str(manifest_path),
 'status':'PILOT_NOT_PRODUCTION' if not PRODUCTION_MODE else ('PREPARATION_COMPLETE_REVIEW_REQUIRED' if complete else 'INCOMPLETE')
}
plan_path=manifest_dir/('sampler_plan_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'.json')
with plan_path.open('x',encoding='utf-8') as f: json.dump(plan,f,indent=2,ensure_ascii=False)
print(json.dumps(plan,indent=2,ensure_ascii=False))
print('Plan:',plan_path)

## Canary readiness report

In [ ]:
# Canary metrics and readiness report — derived from committed SQLite state.
from collections import defaultdict
rows=con.execute('SELECT source,COUNT(*),SUM(docs),SUM(duplicates),SUM(tokens) FROM completed_chunk GROUP BY source ORDER BY source').fetchall()
print('SOURCE | CHUNKS | DOCS | DUPLICATES | TOKENS')
for row in rows: print(*row,sep=' | ')
completed=sum(r[1] for r in rows)
tokens=sum(r[4] for r in rows)
# Report wall-clock session separately from estimated full-corpus throughput.
elapsed_session=time.perf_counter()-started
rate=tokens/elapsed_session if elapsed_session>0 and new_chunks==60 else None
print('Committed chunks:',completed,'/60')
print('Committed tokens:',f'{tokens:,}')
print('Fresh-run effective tokens/sec (only if 60 newly processed):',round(rate,1) if rate else 'N/A — resumed or partial session')
assert completed == 60
assert con.execute('PRAGMA quick_check').fetchone()[0] == 'ok'
verify_committed_shards()
# Verify no committed path lies outside the canary tree.
for (p,) in con.execute('SELECT path FROM shard'):
    assert Path(p).resolve().is_relative_to(OUT.resolve()), 'Shard outside canary output!'
report={'audit':'Maia Corpus v8 production canary v1','status':'CANARY_COMPLETED_REVIEW_REQUIRED',
        'timestamp_utc':datetime.now(timezone.utc).isoformat(),'output':str(OUT),
        'chunks_completed':completed,'tokens_committed':tokens,'documents':sum(r[2] for r in rows),
        'duplicates':sum(r[3] for r in rows),'by_source':[dict(source=r[0],chunks=r[1],docs=r[2],duplicates=r[3],tokens=r[4]) for r in rows],
        'effective_tokens_per_second_fresh_session':rate,
        'limitations':['Drive performance and interruption resilience need operational review',
                       'No full freeze SHA recheck','No near-duplicate check','No license audit',
                       'This canary is not the full production corpus']}
reports=OUT/'reports';reports.mkdir(exist_ok=True)
name='canary_report_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'.json'
p=reports/name
with p.open('x',encoding='utf-8') as f: json.dump(report,f,indent=2,ensure_ascii=False)
print(json.dumps(report,indent=2,ensure_ascii=False))
print('CANARY REPORT:',p)


## Interpretation and production gate

A successful canary demonstrates processing of 60 complete source chunks and validates committed shard integrity. **Do not simply flip this notebook into full production:** the inventory is deliberately capped at 60. Full production must use a **new** immutable output directory and configuration, with a separately reviewed full-inventory plan. Keep all previous pilot and canary directories intact. Share the `canary_report_*.json` and any runtime errors for review.